# Exploratory Data Analysis — Ames Housing

## Part A: Dataset Understanding & Data Quality

This section examines the structure, variable types,
descriptive statistics, missing values, and potential
outliers in the Ames Housing dataset.

In [3]:

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Path relative to the notebook directory
DATA_PATH = Path("data/AmesHousing.csv")

# The provided dataset is tab-separated
df = pd.read_csv(DATA_PATH, sep="\t")

print("Dataset shape:", df.shape)
display(df.head())


Dataset shape: (2930, 82)


,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


# Dataset dimensions

In [4]:
n_rows, n_columns = df.shape

print(f"Number of observations: {n_rows}")
print(f"Number of features/columns: {n_columns}")
print(f"Total number of cells: {df.size}")


Number of observations: 2930
Number of features/columns: 82
Total number of cells: 240260


# Statistical Feature Classification

In [6]:

feature_types = {
    "Nominal": [
        "MS SubClass", "MS Zoning", "Street", "Alley",
        "Land Contour", "Lot Config", "Neighborhood",
        "Condition 1", "Condition 2", "Bldg Type",
        "House Style", "Roof Style", "Roof Matl",
        "Exterior 1st", "Exterior 2nd", "Mas Vnr Type",
        "Foundation", "Heating", "Central Air",
        "Garage Type", "Misc Feature",
        "Sale Type", "Sale Condition"
    ],
    "Ordinal": [
        "Lot Shape", "Utilities", "Land Slope",
        "Overall Qual", "Overall Cond",
        "Exter Qual", "Exter Cond",
        "Bsmt Qual", "Bsmt Cond", "Bsmt Exposure",
        "BsmtFin Type 1", "BsmtFin Type 2",
        "Heating QC", "Electrical", "Kitchen Qual",
        "Functional", "Fireplace Qu",
        "Garage Finish", "Garage Qual", "Garage Cond",
        "Paved Drive", "Pool QC", "Fence"
    ],
    "Discrete": [
        "Year Built", "Year Remod/Add",
        "Bsmt Full Bath", "Bsmt Half Bath",
        "Full Bath", "Half Bath", "Bedroom AbvGr",
        "Kitchen AbvGr", "TotRms AbvGrd",
        "Fireplaces", "Garage Yr Blt", "Garage Cars",
        "Mo Sold", "Yr Sold"
    ],
    "Continuous": [
        "Lot Frontage", "Lot Area", "Mas Vnr Area",
        "BsmtFin SF 1", "BsmtFin SF 2",
        "Bsmt Unf SF", "Total Bsmt SF",
        "1st Flr SF", "2nd Flr SF", "Low Qual Fin SF",
        "Gr Liv Area", "Garage Area",
        "Wood Deck SF", "Open Porch SF",
        "Enclosed Porch", "3Ssn Porch",
        "Screen Porch", "Pool Area", "Misc Val"
    ]
}

identifier_columns = ["Order", "PID"]
target_column = "SalePrice"


In [7]:

from collections import Counter

classified_columns = [
    col
    for columns in feature_types.values()
    for col in columns
]

all_assigned = (
    classified_columns
    + identifier_columns
    + [target_column]
)

missing_from_classification = sorted(
    set(df.columns) - set(all_assigned)
)

unknown_columns = sorted(
    set(all_assigned) - set(df.columns)
)

duplicates = [
    col for col, count in Counter(all_assigned).items()
    if count > 1
]

print("Feature counts by statistical type:")
for feature_type, columns in feature_types.items():
    print(f"{feature_type}: {len(columns)}")

print("\nUnclassified columns:", missing_from_classification)
print("Unknown column names:", unknown_columns)
print("Duplicate assignments:", duplicates)

assert not missing_from_classification
assert not unknown_columns
assert not duplicates

print("\nAll dataset columns classified successfully!")

Feature counts by statistical type:
Nominal: 23
Ordinal: 23
Discrete: 14
Continuous: 19

Unclassified columns: []
Unknown column names: []
Duplicate assignments: []

All dataset columns classified successfully!


In [11]:

classification_table = pd.DataFrame(
    [
        {"Feature": column, "Statistical Type": feature_type}
        for feature_type, columns in feature_types.items()
        for column in columns
    ]
)

classification_table = classification_table.sort_values(
    ["Statistical Type", "Feature"]
).reset_index(drop=True)


for feature_type in feature_types:
    print(f"\n--- {feature_type.upper()} ---")

    display(
        classification_table[
            classification_table["Statistical Type"] == feature_type
        ]
    )




--- NOMINAL ---


,Feature,Statistical Type
33,Alley,Nominal
34,Bldg Type,Nominal
35,Central Air,Nominal
36,Condition 1,Nominal
37,Condition 2,Nominal
38,Exterior 1st,Nominal
39,Exterior 2nd,Nominal
40,Foundation,Nominal
41,Garage Type,Nominal
42,Heating,Nominal



--- ORDINAL ---


,Feature,Statistical Type
56,Bsmt Cond,Ordinal
57,Bsmt Exposure,Ordinal
58,Bsmt Qual,Ordinal
59,BsmtFin Type 1,Ordinal
60,BsmtFin Type 2,Ordinal
61,Electrical,Ordinal
62,Exter Cond,Ordinal
63,Exter Qual,Ordinal
64,Fence,Ordinal
65,Fireplace Qu,Ordinal



--- DISCRETE ---


,Feature,Statistical Type
19,Bedroom AbvGr,Discrete
20,Bsmt Full Bath,Discrete
21,Bsmt Half Bath,Discrete
22,Fireplaces,Discrete
23,Full Bath,Discrete
24,Garage Cars,Discrete
25,Garage Yr Blt,Discrete
26,Half Bath,Discrete
27,Kitchen AbvGr,Discrete
28,Mo Sold,Discrete



--- CONTINUOUS ---


,Feature,Statistical Type
0,1st Flr SF,Continuous
1,2nd Flr SF,Continuous
2,3Ssn Porch,Continuous
3,Bsmt Unf SF,Continuous
4,BsmtFin SF 1,Continuous
5,BsmtFin SF 2,Continuous
6,Enclosed Porch,Continuous
7,Garage Area,Continuous
8,Gr Liv Area,Continuous
9,Lot Area,Continuous


## Interpretation

The Ames Housing dataset contains four main statistical feature types: nominal, ordinal, discrete, and continuous.

- Nominal variables describe unordered categories, such as neighborhood and building type.
- Ordinal variables represent ordered evaluations, such as overall quality and exterior condition.
- Discrete variables include counts of rooms and bathrooms, as well as recorded years and months.
- Continuous variables describe measurable properties, such as living area and lot size.

The classification is based on the dataset's data dictionary.
Identifier columns (`Order` and `PID`) are treated separately,
while `SalePrice` is identified as the target variable.

## Descriptive Statistics

This section summarizes the statistical properties of the Ames Housing dataset according to the semantic feature types.

Different descriptive measures are used for nominal, ordinal, discrete, and continuous features.

In [12]:

# A4.1 — Nominal features

nominal_summary = pd.DataFrame([
    {
        "Feature": col,
        "Unique Categories": df[col].nunique(dropna=True),
        "Most Frequent Category": (
            df[col].mode(dropna=True).iloc[0]
            if df[col].notna().any() else np.nan
        ),
        "Mode Frequency": (
            df[col].value_counts().iloc[0]
            if df[col].notna().any() else 0
        ),
        "Mode Percentage": (
            df[col].value_counts(normalize=True).iloc[0] * 100
            if df[col].notna().any() else np.nan
        )
    }
    for col in feature_types["Nominal"]
])

display(nominal_summary.round(2))

,Feature,Unique Categories,Most Frequent Category,Mode Frequency,Mode Percentage
0,MS SubClass,16,20,1079,36.83
1,MS Zoning,7,RL,2273,77.58
2,Street,2,Pave,2918,99.59
3,Alley,2,Grvl,120,60.61
4,Land Contour,4,Lvl,2633,89.86
5,Lot Config,5,Inside,2140,73.04
6,Neighborhood,28,NAmes,443,15.12
7,Condition 1,9,Norm,2522,86.08
8,Condition 2,8,Norm,2900,98.98
9,Bldg Type,5,1Fam,2425,82.76


In [13]:

# Neighborhood distribution

neighborhood_distribution = pd.DataFrame({
    "Count": df["Neighborhood"].value_counts(),
    "Percentage": (
        df["Neighborhood"].value_counts(normalize=True) * 100
    ).round(2)
})

display(neighborhood_distribution)


,Count,Percentage
Neighborhood,,
NAmes,443,15.12
CollgCr,267,9.11
OldTown,239,8.16
Edwards,194,6.62
Somerst,182,6.21
NridgHt,166,5.67
Gilbert,165,5.63
Sawyer,151,5.15
NWAmes,131,4.47


In [14]:

# A4.2 — Ordinal features
# Frequency distribution of overall quality

overall_quality_distribution = (
    df["Overall Qual"]
    .value_counts(dropna=False)
    .sort_index()
)

display(overall_quality_distribution)

print(
    "Median overall quality:",
    df["Overall Qual"].median()
)

print(
    "Most frequent quality rating:",
    df["Overall Qual"].mode().iloc[0]
)


Overall Qual
1       4
2      13
3      40
4     226
5     825
6     732
7     602
8     350
9     107
10     31
Name: count, dtype: int64

Median overall quality: 6.0
Most frequent quality rating: 5


In [15]:

# A4.3 — Discrete count features

discrete_count_features = [
    "Bedroom AbvGr",
    "Full Bath",
    "Half Bath",
    "Fireplaces",
    "Garage Cars",
    "TotRms AbvGrd"
]

discrete_summary = df[
    discrete_count_features
].describe().T

display(discrete_summary.round(2))

,count,mean,std,min,25%,50%,75%,max
Bedroom AbvGr,2930.0,2.85,0.83,0.0,2.0,3.0,3.0,8.0
Full Bath,2930.0,1.57,0.55,0.0,1.0,2.0,2.0,4.0
Half Bath,2930.0,0.38,0.50,0.0,0.0,0.0,1.0,2.0
Fireplaces,2930.0,0.60,0.65,0.0,0.0,1.0,1.0,4.0
Garage Cars,2929.0,1.77,0.76,0.0,1.0,2.0,2.0,5.0
TotRms AbvGrd,2930.0,6.44,1.57,2.0,5.0,6.0,7.0,15.0


In [16]:

# A4.4 — Continuous features

continuous_summary = df[
    feature_types["Continuous"]
].describe().T

display(continuous_summary.round(2))


,count,mean,std,min,25%,50%,75%,max
Lot Frontage,2440.0,69.22,23.37,21.0,58.00,68.0,80.00,313.0
Lot Area,2930.0,10147.92,7880.02,1300.0,7440.25,9436.5,11555.25,215245.0
Mas Vnr Area,2907.0,101.90,179.11,0.0,0.00,0.0,164.00,1600.0
BsmtFin SF 1,2929.0,442.63,455.59,0.0,0.00,370.0,734.00,5644.0
BsmtFin SF 2,2929.0,49.72,169.17,0.0,0.00,0.0,0.00,1526.0
Bsmt Unf SF,2929.0,559.26,439.49,0.0,219.00,466.0,802.00,2336.0
Total Bsmt SF,2929.0,1051.61,440.62,0.0,793.00,990.0,1302.00,6110.0
1st Flr SF,2930.0,1159.56,391.89,334.0,876.25,1084.0,1384.00,5095.0
2nd Flr SF,2930.0,335.46,428.40,0.0,0.00,0.0,703.75,2065.0
Low Qual Fin SF,2930.0,4.68,46.31,0.0,0.00,0.0,0.00,1064.0


In [17]:

# A4.5 — Target variable

sale_price_summary = df["SalePrice"].describe()

display(sale_price_summary.round(2))

print(
    "SalePrice skewness:",
    round(df["SalePrice"].skew(), 3)
)

count      2930.00
mean     180796.06
std       79886.69
min       12789.00
25%      129500.00
50%      160000.00
75%      213500.00
max      755000.00
Name: SalePrice, dtype: float64

SalePrice skewness: 1.744
